In [1]:
import sys

sys.path.append("/Users/krisztianivankai/PycharmProjects/Szakdolgozat/")
from forecast.forecast import run_experiment_fuzzy_and_average_model
from utilities.utils import *

In [2]:
sales_and_stock = pd.read_csv('Szakdoga_adat.csv')
split_percentage = 0.8
lags_to_use = 2

In [3]:
get_date_features(sales_and_stock)

,date,sales,stock,day_of_week,month,quarter,year,year_quarter
0,2024-01-01,NaN,32,1,1,1,2024,202401
1,2024-01-02,10.0,22,2,1,1,2024,202401
2,2024-01-03,13.0,9,3,1,1,2024,202401
3,2024-01-04,25.0,34,4,1,1,2024,202401
4,2024-01-05,27.0,7,5,1,1,2024,202401
...,...,...,...,...,...,...,...,...
784,2026-02-23,76.0,335,1,2,1,2026,202601
785,2026-02-24,74.0,261,2,2,1,2026,202601
786,2026-02-25,66.0,195,3,2,1,2026,202601
787,2026-02-26,55.0,140,4,2,1,2026,202601


In [4]:
demand = get_prepared_demand_df(sales_and_stock)

In [5]:
split_date = get_split_date(sales_and_stock['date'], split_percentage)
train_data = demand[demand['date'] < split_date].reset_index(drop=True)

In [6]:
statistics = train_data[['demand']].describe()

In [7]:
demand_min = 0 #statistics.T["min"].to_list()[0]
demand_first_quartile = 25#statistics.T["25%"].to_list()[0]
demand_median = 50#statistics.T["50%"].to_list()[0]
demand_third_quartile = 100#statistics.T["75%"].to_list()[0]
demand_max = 500#statistics.T["max"].to_list()[0]

In [8]:
fuzzy_sets = [
    {
        "name": "VeryLowDemand",
        "type": "shoulder",
        "a": demand_min,
        "b": demand_first_quartile,
        "direction": "left",
    },
    {
        "name": "LowDemand",
        "type": "triangular",
        "a": demand_min,
        "b": demand_first_quartile,
        "c": demand_median,
    },
    {
        "name": "MediumDemand",
        "type": "triangular",
        "a": demand_first_quartile,
        "b": demand_median,
        "c": demand_third_quartile,
    },
    {
        "name": "HighDemand",
        "type": "triangular",
        "a": demand_median,
        "b": demand_third_quartile,
        "c": demand_max
    },
    {
        "name": "VeryHighDemand",
        "type": "shoulder",
        "a": demand_third_quartile,
        "b": demand_max,
        "direction": "right"
    }
]

In [9]:
fuzzy_sets = [
    {
        "name": "VeryLowDemand",
        "type": "shoulder",
        "a": demand_min,
        "b": demand_first_quartile,
        "direction": "left",
    },
    {
        "name": "MediumDemand",
        "type": "triangular",
        "a": demand_first_quartile,
        "b": demand_median,
        "c": demand_third_quartile,
    },
    {
        "name": "VeryHighDemand",
        "type": "shoulder",
        "a": demand_third_quartile,
        "b": demand_max,
        "direction": "right"
    }
]

In [10]:
for lags_to_use in range(1,30):
    fuzzy_score, average_score = run_experiment_fuzzy_and_average_model(demand, split_date, lags_to_use, fuzzy_sets)

    params_and_results = [
        {
            "lags_to_use": lags_to_use,
            "fuzzy_sets": fuzzy_sets,
            "fuzzy_score": round(float(fuzzy_score.score.max()),4),
            "average_score": round(float(average_score.score.max()),4),
        }
    ]

    try:
        new_run = pd.DataFrame(params_and_results)
        existing_runs = pd.read_csv('experiment_runs.csv')
        pd.concat([existing_runs, new_run]).to_csv('experiment_runs.csv', index=False)
    except:
        pd.DataFrame(params_and_results).to_csv('experiment_runs.csv', index=False)

In [11]:
all_runs = pd.read_csv('experiment_runs.csv').sort_values(['fuzzy_score'])


In [12]:
all_runs.head()

,lags_to_use,fuzzy_sets,fuzzy_score,average_score
6,7,"[{'name': 'VeryLowDemand', 'type': 'shoulder',...",0.3388,0.4141
7,8,"[{'name': 'VeryLowDemand', 'type': 'shoulder',...",0.3536,0.4188
8,9,"[{'name': 'VeryLowDemand', 'type': 'shoulder',...",0.3700,0.4225
5,6,"[{'name': 'VeryLowDemand', 'type': 'shoulder',...",0.3755,0.4205
30,2,"[{'name': 'VeryLowDemand', 'type': 'shoulder',...",0.3759,0.4717


In [13]:
all_runs.head(1)

,lags_to_use,fuzzy_sets,fuzzy_score,average_score
6,7,"[{'name': 'VeryLowDemand', 'type': 'shoulder',...",0.3388,0.4141
